# Assignment 2 — Exploratory Data Analysis
### Retail Sales Dataset  |  100 Marks  |  6 Tasks

---

**Prerequisite:** Run `A1_Data_Cleaning.ipynb` first — this notebook loads `retail_sales_clean.csv`.

| Assignment | Notebook |
|---|---|
| Assignment 1 | Data Cleaning |
| Assignment 2 | Exploratory Data Analysis |
| Assignment 3 | Statistical Analysis |
| Assignment 4 | Segmentation and Clustering |
| Assignment 5 | Time Series Forecasting |


## Environment Setup

In [ ]:
# Run once to install any missing packagesimport subprocess, syspkgs = ['pandas','numpy','matplotlib','seaborn','scipy','statsmodels','scikit-learn','pmdarima','prophet']for p in pkgs:subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)print("All packages ready.")

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# Plot stylesns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(12,5),'axes.titlesize':13, 'axes.labelsize':11})print("Imports complete ")

# Assignment 2 — Exploratory Data Analysis (EDA)
**Marks: 100  |  Tasks: 6**

---

## What is EDA?

Exploratory Data Analysis (EDA), introduced by statistician John Tukey in 1977, is the process of using visual and quantitative methods to understand the structure, patterns, relationships, and anomalies in a dataset *before* applying formal statistical tests or building models.

EDA is a philosophy as much as a technique: approach your data with curiosity and skepticism. Let the data tell you what is interesting rather than confirming pre-existing hypotheses.

### Why EDA Comes Before Modeling

Every predictive model makes assumptions about the data (linearity, normality, independence). EDA reveals whether those assumptions hold. Skipping EDA and going straight to modeling risks:
- Building a model on data with undetected outliers that dominate the coefficients
- Treating a non-linear relationship as linear, producing biased predictions
- Missing a confounding variable that explains most of the outcome
- Presenting findings that contradict obvious patterns visible in the data

### The EDA Workflow (6 Tasks)

| Task | Level | What You Learn |
|---|---|---|
| 1 | Univariate (1 variable) | Distribution shape, central tendency, spread |
| 2 | Univariate visual | Histograms, box plots, count plots |
| 3 | Bivariate (2 variables) | Relationships, correlations, group differences |
| 4 | Time dimension | Trends, seasonality, growth rates |
| 5 | Multivariate (3+ variables) | Interaction effects, clusters, correlations |
| 6 | Business insights | Actionable findings, recommendations |

### Visualization Principles

Before coding, remember these design principles:
- **Every chart needs a title, axis labels, and units**
- **Sort categories by value** (not alphabetically) for bar charts — it makes comparisons instantly easier
- **Use consistent color schemes** across related charts
- **Annotate the most important feature** in each chart (peak value, threshold line, trend direction)
- **A chart without an interpretation is incomplete** — always write what the chart shows


In [ ]:
# Load the cleaned datasetimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsimport warningswarnings.filterwarnings('ignore')sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)plt.rcParams.update({'figure.dpi':110, 'axes.titlesize':13})df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])# Re-cast categoricals (lost on CSV round-trip)for col in ['product_category','region','gender','payment_method','order_status']:df[col] = df[col].astype('category')print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")print(df.dtypes.to_string())

---
## Task 1 — Descriptive Statistics

### Theory: The Five-Number Summary and Beyond

Descriptive statistics reduce a column of potentially thousands of values into a small set of numbers that capture its essential character. Pandas `.describe()` computes the five-number summary (min, Q1, median, Q3, max) plus mean, std, and count.

### Measures of Central Tendency

| Measure | Formula | Sensitivity to Outliers | Best For |
|---|---|---|---|
| Mean | sum(x) / n | High — a single extreme value shifts the mean | Symmetric, outlier-free distributions |
| Median | middle value when sorted | None — unaffected by extreme values | Skewed distributions, income, revenue |
| Mode | most frequent value | None | Categorical variables |

**Key diagnostic:** If `mean >> median`, the distribution has a long right tail (positive skew). This is typical for revenue, order value, and waiting times. Always report the median for skewed financial data.

### Measures of Spread

| Measure | Formula | Notes |
|---|---|---|
| Range | max - min | Highly sensitive to outliers |
| Variance | sum((xi - mean)^2) / (n-1) | Squared units — harder to interpret |
| Standard deviation | sqrt(variance) | Same units as the variable — most interpretable |
| IQR | Q3 - Q1 | Robust spread measure; covers middle 50% |
| Coefficient of Variation | std / mean | Normalized spread; enables comparison across variables |

### Skewness and Kurtosis

**Skewness** measures the asymmetry of the distribution:
- Skewness = 0: perfectly symmetric (normal distribution)
- Skewness > 1: right-skewed (long right tail) — common for revenue, time
- Skewness < -1: left-skewed (long left tail)

**Kurtosis** (excess kurtosis) measures tail heaviness relative to a normal distribution:
- Kurtosis = 0: normal tails
- Kurtosis > 3: heavy tails (leptokurtic) — more extreme values than expected
- Kurtosis < 0: light tails (platykurtic)

High kurtosis in a residual distribution is a red flag for regression models — it means extreme errors are more common than the normal distribution predicts.


In [ ]:
numeric_cols = ['sales_amount','profit','age','discount_pct','days_to_ship','quantity','unit_price','shipping_cost','customer_satisfaction']# Full describeprint("=== df.describe() ===")print(df[numeric_cols].describe().round(2).to_string())

In [ ]:
# Skewness, kurtosis, CoV summary tablesummary = pd.DataFrame({'Mean': df[numeric_cols].mean(),'Median': df[numeric_cols].median(),'Std': df[numeric_cols].std(),'Min': df[numeric_cols].min(),'Max': df[numeric_cols].max(),'IQR': df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),'Skewness': df[numeric_cols].skew(),'Kurtosis': df[numeric_cols].kurt(),'CoV %': (df[numeric_cols].std() / df[numeric_cols].mean() * 100),}).round(3)print("=== Extended Summary Statistics ===")print(summary.to_string())print("\nMost right-skewed (high CoV):", summary['CoV %'].idxmax())print("Highest kurtosis (heaviest tails):", summary['Kurtosis'].idxmax())

In [ ]:
# Visual: sorted skewness bar chartfig, axes = plt.subplots(1, 2, figsize=(14, 5))summary['Skewness'].sort_values().plot(kind='barh', color=['#e74c3c' if v>0 else '#2ecc71'for v in summary['Skewness'].sort_values()], ax=axes[0])axes[0].axvline(0, color='black', linewidth=1)axes[0].set_title('Skewness per Column\n(red = right-skewed, green = left-skewed)')axes[0].set_xlabel('Skewness')summary['CoV %'].sort_values(ascending=False).plot(kind='barh', color='steelblue', ax=axes[1])axes[1].set_title('Coefficient of Variation %\n(higher = more spread relative to mean)')axes[1].set_xlabel('CoV %')plt.tight_layout()plt.show()

---
## Task 2 — Univariate Analysis

### Theory: Visualizing Single Variables

Univariate analysis examines one variable at a time to understand its distribution, central tendency, spread, and shape. The choice of visualization depends on whether the variable is continuous (numeric) or discrete (categorical).

### For Continuous Variables

**Histogram:** Divides the range into equal-width bins and counts observations per bin.
- Bin width matters — too few bins (e.g., 5) hides structure; too many (e.g., 200) creates noise
- Rule of thumb for bin count: sqrt(n) or Sturges' formula: k = 1 + 3.322 * log10(n)
- A histogram is the most honest visualization of a distribution

**KDE (Kernel Density Estimate):** A smoothed continuous estimate of the probability density function. Useful for comparing distributions between groups on the same axes. The bandwidth parameter controls smoothness — a very small bandwidth creates spiky curves; very large creates over-smoothed curves.

**Box Plot:** Displays the five-number summary (Q1, Q2/median, Q3, min, max) plus outliers as individual points. Excellent for comparing distributions across multiple groups side-by-side.

**Violin Plot:** Combines a box plot with a mirrored KDE curve on each side. Shows more information than a box plot (bimodality, skewness) at the cost of being slightly harder to read.

### For Categorical Variables

**Count Plot / Bar Chart:** Shows the frequency of each category. Always sort by count (descending) for readability. Rotate x-axis labels if they overlap.

**Pie Chart:** Shows proportional composition. Only use when there are 5 or fewer categories; more slices become unreadable.

### Reading a Box Plot

```
     |-----|=========|=====|-----|   o  o
     Min   Q1     Median  Q3   Max  (outliers)
                  |IQR = Q3-Q1|
```

- The box spans Q1 to Q3 (IQR — the middle 50% of the data)
- The line inside the box is the median
- Whiskers extend to the most extreme value within 1.5 * IQR from the box
- Points beyond the whiskers are plotted individually as outliers


In [ ]:
# Histograms with KDEfig, axes = plt.subplots(2, 2, figsize=(14, 9))cols_hist = ['sales_amount','profit','age','discount_pct']colors = ['#3498db','#2ecc71','#e67e22','#9b59b6']for ax, col, color in zip(axes.flat, cols_hist, colors):sns.histplot(df[col].dropna(), kde=True, ax=ax, color=color, bins=40,line_kws={'linewidth':2})ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean {df[col].mean():.0f}')ax.axvline(df[col].median(), color='black', linestyle=':', label=f'Median {df[col].median():.0f}')ax.set_title(f'Distribution of {col}')ax.legend(fontsize=9)ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))plt.suptitle('Histograms with KDE — Red dashed=Mean, Black dotted=Median', fontsize=12, y=1.01)plt.tight_layout()plt.show()print("\nObservation: sales_amount and profit are right-skewed (mean > median).")print("This means a few very large orders are pulling the average up.")

In [ ]:
# Boxplots gridfig, axes = plt.subplots(2, 4, figsize=(16, 7))box_cols = ['sales_amount','profit','age','discount_pct','quantity','days_to_ship','unit_price','shipping_cost']for ax, col in zip(axes.flat, box_cols):bp = df.boxplot(column=col, ax=ax,boxprops=dict(color='steelblue'),medianprops=dict(color='red', linewidth=2),flierprops=dict(marker='o', color='orange', markersize=3, alpha=0.5))q1, med, q3 = df[col].quantile([0.25,0.5,0.75])ax.set_title(f'{col}\nMedian={med:,.0f} IQR={q3-q1:,.0f}', fontsize=9)ax.set_xlabel('')plt.suptitle('Boxplots — Orange dots are outliers', fontsize=13, y=1.01)plt.tight_layout()plt.show()

In [ ]:
# Bar charts for categorical columnscat_cols = ['product_category','region','payment_method','gender','order_status']fig, axes = plt.subplots(2, 3, figsize=(16, 9))palette = sns.color_palette('Set2', 10)for ax, col in zip(axes.flat, cat_cols):counts = df[col].value_counts()counts.sort_values().plot(kind='barh', ax=ax, color=palette[:len(counts)])ax.set_title(f'{col} — Order Count')ax.set_xlabel('Count')for i, v in enumerate(counts.sort_values()):ax.text(v + 5, i, f'{v:,}', va='center', fontsize=9)axes.flat[-1].set_visible(False)plt.suptitle('Categorical Column Distributions', fontsize=13, y=1.01)plt.tight_layout()plt.show()

In [ ]:
# Pie chart: return rateret_counts = df['return_flag'].value_counts()labels = ['Not Returned', 'Returned']colors = ['#2ecc71', '#e74c3c']explode = [0, 0.08]fig, ax = plt.subplots(figsize=(6, 6))wedges, texts, autotexts = ax.pie(ret_counts, labels=labels, colors=colors,autopct='%1.1f%%', explode=explode,startangle=90, textprops={'fontsize':12})autotexts[0].set_fontsize(14)autotexts[1].set_fontsize(14)ax.set_title(f'Return Rate\nTotal Orders: {len(df):,}', fontsize=13)plt.tight_layout()plt.show()print(f"Return rate: {df['return_flag'].mean()*100:.1f}%")

---
## Task 3 — Bivariate Analysis

### Theory: Examining Relationships Between Two Variables

Bivariate analysis examines how two variables relate to each other. The appropriate technique depends on the types of both variables:

| Variable 1 | Variable 2 | Technique | Chart |
|---|---|---|---|
| Continuous | Continuous | Pearson/Spearman correlation | Scatter plot, regression plot |
| Categorical | Continuous | Group statistics, t-test/ANOVA | Box plot, violin plot, grouped bar |
| Categorical | Categorical | Chi-square, cross-tabulation | Grouped bar, stacked bar, heatmap |

### Scatter Plot — The Workhorse of Bivariate Analysis

A scatter plot places a point at (x, y) for each observation. The pattern reveals the nature of the relationship:

- **Positive linear:** Both variables increase together (upward diagonal pattern)
- **Negative linear:** One increases as the other decreases (downward diagonal)
- **Non-linear:** Curved pattern — quadratic, logarithmic, exponential
- **No relationship:** Cloud of points with no directional pattern
- **Heteroscedastic:** The spread of y values increases with x (funnel shape) — problematic for regression

**Always add a regression line** to confirm the direction and rough slope of the relationship.

### Interpreting Correlation Visually Before Computing It

Before calculating Pearson r, look at the scatter plot:
- If the scatter is tight around a line, |r| is close to 1
- If it is loosely distributed, |r| is close to 0
- If there is a clear curved (but not linear) relationship, Pearson r will understate the relationship — use Spearman rank correlation instead

### Group Comparisons (Categorical vs Continuous)

The box plot is the standard tool for comparing a numeric distribution across multiple categories:
- Sort categories by their median value (not alphabetically) to make the ranking immediately visible
- Overlapping boxes suggest groups are similar; non-overlapping medians suggest real differences
- Wide IQR (tall box) means high within-group variability; narrow IQR means consistency

### Simpson's Paradox

A critical warning for bivariate analysis: a trend observed in the aggregate data can reverse when you control for a third variable. For example, a discount might appear to reduce revenue overall, but within each product category, it increases revenue — the aggregate reversal is caused by the mix of products in the discount group. Always check bivariate relationships across subgroups before drawing conclusions.


In [ ]:
# Scatter: sales_amount vs profit, coloured by product_categoryfig, ax = plt.subplots(figsize=(12, 6))categories = df['product_category'].cat.categoriespalette = sns.color_palette('tab10', len(categories))for cat, color in zip(categories, palette):mask = df['product_category'] == catax.scatter(df.loc[mask,'sales_amount'], df.loc[mask,'profit'],alpha=0.35, s=18, color=color, label=cat)ax.set_xlabel('Sales Amount (INR)')ax.set_ylabel('Profit (INR)')ax.set_title('Sales Amount vs Profit — coloured by Product Category')ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))plt.tight_layout()plt.show()corr = df['sales_amount'].corr(df['profit'])print(f"Pearson r (sales vs profit): {corr:.3f} — {'strong positive' if corr>0.7 else 'moderate'} relationship")

In [ ]:
# Box: sales_amount by regionfig, ax = plt.subplots(figsize=(10, 5))region_order = df.groupby('region')['sales_amount'].median().sort_values(ascending=False).indexdf.boxplot(column='sales_amount', by='region', ax=ax,order=region_order,boxprops=dict(color='steelblue'),medianprops=dict(color='red', linewidth=2.5),flierprops=dict(marker='.', alpha=0.3))# Annotate mediansfor i, region in enumerate(region_order):med = df.loc[df['region']==region, 'sales_amount'].median()ax.text(i+1, med*1.03, f'₹{med:,.0f}', ha='center', fontsize=9, color='red')ax.set_title('Sales Amount Distribution by Region')ax.set_xlabel('Region'); ax.set_ylabel('Sales Amount (INR)')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))plt.suptitle('')plt.tight_layout()plt.show()print("Median sales by region:")print(df.groupby('region')['sales_amount'].median().sort_values(ascending=False).apply(lambda x: f'₹{x:,.0f}'))

In [ ]:
# Bar: avg profit margin by product_categorydf['profit_margin_pct'] = df['profit'] / df['sales_amount'] * 100margin_by_cat = df.groupby('product_category')['profit_margin_pct'].mean().sort_values(ascending=False)fig, ax = plt.subplots(figsize=(10, 5))colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in margin_by_cat]bars = margin_by_cat.plot(kind='bar', ax=ax, color=colors, edgecolor='white')ax.axhline(0, color='black', linewidth=1)ax.set_title('Average Profit Margin % by Product Category')ax.set_xlabel('Category'); ax.set_ylabel('Avg Profit Margin %')ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')for i, v in enumerate(margin_by_cat):ax.text(i, v + 0.5 if v >= 0 else v - 2, f'{v:.1f}%', ha='center', fontsize=10)plt.tight_layout()plt.show()print(f"Most profitable category: {margin_by_cat.idxmax()} ({margin_by_cat.max():.1f}%)")print(f"Least profitable category: {margin_by_cat.idxmin()} ({margin_by_cat.min():.1f}%)")

In [ ]:
# Scatter: discount_pct vs sales_amountfrom scipy import stats as sp_statsfig, ax = plt.subplots(figsize=(10, 5))ax.scatter(df['discount_pct'], df['sales_amount'], alpha=0.2, s=15, color='#9b59b6')# Regression lineslope, intercept, r, p, _ = sp_stats.linregress(df['discount_pct'], df['sales_amount'])x_line = np.linspace(0, 0.4, 100)ax.plot(x_line, slope*x_line + intercept, color='red', linewidth=2,label=f'r = {r:.3f} (p = {p:.4f})')ax.set_xlabel('Discount %'); ax.set_ylabel('Sales Amount (INR)')ax.set_title('Discount % vs Sales Amount — Does offering a bigger discount drive higher sales?')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))ax.legend()plt.tight_layout()plt.show()print(f"r = {r:.3f} → {'positive' if r>0 else 'negative'} relationship")print("Interpretation: " + ("Higher discounts correlate with higher sales value." if r>0else "Higher discounts do NOT drive higher sales — interesting!"))

---
## Task 4 — Time Series and Trend Analysis

### Theory: Analyzing Data Over Time

Time series analysis examines how a variable changes over ordered time intervals. Unlike cross-sectional data where rows are independent, time series observations are correlated — today's sales are influenced by yesterday's sales. This temporal dependence is both the main challenge and the main source of predictive power in time series data.

### Four Components of a Time Series

Any time series can be decomposed into four underlying components:

| Component | Definition | Example |
|---|---|---|
| Trend (T) | Long-term direction of the series | Revenue growing 8% per year |
| Seasonality (S) | Regular repeating pattern at a fixed period | Holiday sales spike every December |
| Cyclicality (C) | Irregular multi-year oscillation | Economic boom-bust cycles |
| Irregular/Noise (I) | Random unexplained variation | Day-to-day fluctuation |

**Additive model:** X_t = T + S + C + I  (seasonal amplitude is constant)

**Multiplicative model:** X_t = T x S x C x I  (seasonal amplitude grows with trend level)

Use an additive model when seasonal swings are roughly the same size regardless of the trend level. Use multiplicative when seasonal swings grow proportionally as the series grows — common in retail revenue data.

### Resampling

Raw transactional data must be aggregated to a regular time frequency before time-series analysis:

```python
monthly = df.set_index('order_date')['revenue'].resample('M').sum()
```

Common offset strings: 'D' (day), 'W' (week), 'M' (month end), 'MS' (month start), 'Q' (quarter), 'A' (year).

### Growth Rate Analysis

| Metric | Formula | Use |
|---|---|---|
| Month-over-Month (MoM) | (x_t - x_{t-1}) / x_{t-1} | Short-term momentum |
| Year-over-Year (YoY) | (x_t - x_{t-12}) / x_{t-12} | Removes seasonality; shows true growth |
| Compound Annual Growth Rate (CAGR) | (end/start)^(1/years) - 1 | Long-term average growth rate |

### Rolling Statistics

Rolling (moving) averages smooth out short-term noise to reveal the underlying trend:
- **3-month rolling mean:** Smooths monthly noise
- **12-month rolling mean:** Removes seasonal variation, shows pure trend
- **Rolling std:** Reveals periods of high volatility

When the short-term rolling mean crosses the long-term rolling mean from below, it often signals a trend reversal (a concept borrowed from technical analysis in finance).


In [ ]:
# Set order_date as index and resample monthlydf_ts = df.set_index('order_date').sort_index()monthly = df_ts['sales_amount'].resample('ME').sum()rolling_3m = monthly.rolling(window=3, center=True).mean()fig, ax = plt.subplots(figsize=(14, 5))ax.plot(monthly.index, monthly.values, color='steelblue', alpha=0.6,linewidth=1.5, label='Monthly Sales')ax.plot(rolling_3m.index, rolling_3m.values, color='red', linewidth=2.5,label='3-Month Rolling Mean')ax.fill_between(monthly.index, monthly.values, alpha=0.1, color='steelblue')ax.set_title('Monthly Sales Amount — 2020 to 2024', fontsize=13)ax.set_xlabel('Month'); ax.set_ylabel('Total Sales (INR)')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.legend()plt.tight_layout()plt.show()print(f"Total sales: ₹{monthly.sum()/1e6:.1f}M | Avg monthly: ₹{monthly.mean()/1e6:.2f}M")print(f"Peak month: {monthly.idxmax().strftime('%b %Y')} (₹{monthly.max()/1e6:.2f}M)")

In [ ]:
# Multi-year overlay: compare seasonality across yearsmonthly_df = monthly.reset_index()monthly_df.columns = ['date','sales']monthly_df['year'] = monthly_df['date'].dt.yearmonthly_df['month'] = monthly_df['date'].dt.monthfig, ax = plt.subplots(figsize=(13, 5))colors_yr = {2020:'#e74c3c',2021:'#f39c12',2022:'#2ecc71',2023:'#3498db',2024:'#9b59b6'}month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']for yr, grp in monthly_df.groupby('year'):ax.plot(grp['month'], grp['sales'], marker='o', linewidth=2,color=colors_yr[yr], label=str(yr), markersize=5)ax.set_xticks(range(1,13))ax.set_xticklabels(month_names)ax.set_title('Monthly Sales by Year — Seasonal Pattern Comparison')ax.set_ylabel('Total Sales (INR)')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.legend(title='Year')plt.tight_layout()plt.show()print("Observation: Check if Q4 (Oct-Dec) consistently shows a peak across all years.")

In [ ]:
# Month-over-Month growth ratemom_growth = monthly.pct_change() * 100fig, ax = plt.subplots(figsize=(14, 5))colors_bar = ['#2ecc71' if v >= 0 else '#e74c3c' for v in mom_growth.dropna()]ax.bar(mom_growth.dropna().index, mom_growth.dropna().values,color=colors_bar, width=20, alpha=0.8)ax.axhline(0, color='black', linewidth=1)ax.set_title('Month-over-Month (MoM) Sales Growth %')ax.set_ylabel('Growth %')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))plt.tight_layout()plt.show()print("Top 3 MoM growth months:")print(mom_growth.nlargest(3).apply(lambda x: f'{x:.1f}%'))print("\nTop 5 highest-revenue months:")print(monthly.nlargest(5).apply(lambda x: f'₹{x/1e6:.2f}M'))

---
## Task 5 — Multivariate Analysis

### Theory: Analyzing Three or More Variables Simultaneously

Bivariate analysis reveals pairwise relationships; multivariate analysis examines how multiple variables interact. This is where the most valuable insights live — because real outcomes in business are driven by combinations of factors, not single variables.

### The Correlation Matrix

The correlation matrix is a symmetric n x n table where each cell (i, j) contains the Pearson correlation coefficient between variable i and variable j. The diagonal is always 1.0 (a variable is perfectly correlated with itself).

Visualized as a heatmap with a diverging color scheme (blue for negative, red for positive), it allows instant identification of strongly correlated variable pairs.

**Multicollinearity warning:** If two predictor variables have |r| > 0.8, they carry redundant information. Including both in a regression model will inflate standard errors and make individual coefficient estimates unreliable (even if the model's overall R-squared is fine).

### Pivot Table Heatmap

A pivot table aggregates a numeric metric across two categorical dimensions simultaneously. This reveals interaction effects invisible in univariate or bivariate analysis.

For example, a pivot table of mean revenue by product_category (rows) and region (columns) shows:
- Which product-region combinations generate the highest revenue
- Whether the regional revenue pattern is consistent across all products, or varies by product

### Pair Plot

A pair plot (seaborn `pairplot`) creates a grid of scatter plots for every pair of numeric variables, with univariate distributions on the diagonal. It provides a comprehensive overview of all pairwise relationships in one visualization.

Practical limit: pair plots become hard to read with more than 8 variables. Use feature selection or PCA to reduce dimensionality first.

### Parallel Coordinates Plot

A parallel coordinates plot draws a vertical axis for each variable and connects each observation's values with a line. This allows visual inspection of patterns across many variables simultaneously. Lines that clearly separate by group color indicate that the variable on that axis strongly distinguishes the groups.

### Detecting Interaction Effects

An interaction effect occurs when the relationship between two variables depends on the value of a third variable. For example: discount percentage may have a positive effect on revenue for electronics but a negative effect for clothing. Detecting interactions requires either:
1. Visual inspection of faceted plots (same plot drawn for each sub-group)
2. Two-way ANOVA or interaction terms in regression
3. Tree-based models, which automatically capture interactions


In [ ]:
# Correlation matrix heatmapnum_cols = ['sales_amount','profit','age','discount_pct','quantity','unit_price','days_to_ship','customer_satisfaction','shipping_cost']corr_matrix = df[num_cols].corr()fig, ax = plt.subplots(figsize=(11, 8))mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1) # show lower trianglesns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',center=0, vmin=-1, vmax=1, square=True, ax=ax,linewidths=0.5, mask=mask)ax.set_title('Pearson Correlation Matrix\n(only lower triangle shown)', fontsize=12)plt.tight_layout()plt.show()# Print top correlationscorr_unstacked = corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))top = corr_unstacked.abs().unstack().dropna().sort_values(ascending=False).head(5)print("Top 5 strongest pairwise correlations:")for (c1,c2), v in top.items():print(f" {c1} ↔ {c2}: {corr_matrix.loc[c1,c2]:.3f}")

In [ ]:
# Pivot heatmap: avg sales by region × product_categorypivot = df.pivot_table(values='sales_amount',index='region',columns='product_category',aggfunc='mean')fig, ax = plt.subplots(figsize=(13, 5))sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax,linewidths=0.5, cbar_kws={'label':'Avg Sales (INR)'})ax.set_title('Average Sales Amount (INR) — Region × Product Category')ax.set_xlabel('Product Category'); ax.set_ylabel('Region')plt.tight_layout()plt.show()best_region, best_cat = np.unravel_index(pivot.values.argmax(), pivot.shape)print(f"Highest avg sales: Region={pivot.index[best_region]}, "f"Category={pivot.columns[best_cat]} (₹{pivot.values.max():,.0f})")

---
## Task 6 — Business Insights Report

### Theory: Turning Data into Decisions

EDA is not complete when you have made charts — it is complete when you have extracted actionable insights and communicated them clearly. The gap between "we have data" and "we make better decisions because of this data" is bridged by the Business Insights Report.

### The Insight Pyramid

Each insight should have three layers:

| Layer | Question | Example |
|---|---|---|
| Observation | What does the data show? | "The South region has the highest return rate (18.3%)" |
| Implication | Why does it matter? | "Returns reduce net revenue by approximately 14% in that region" |
| Recommendation | What should be done? | "Investigate product quality or delivery processes in the South; implement a regional quality audit" |

A chart or table without an interpretation is just data. An observation without an implication is just a fact. An implication without a recommendation is just analysis. Only all three together constitute an actionable insight.

### Types of Business Insights from EDA

1. **Revenue concentration:** Pareto analysis — which 20% of products/customers/regions generate 80% of revenue?
2. **Underperforming segments:** Which product-region combinations consistently underperform the portfolio average?
3. **Seasonality patterns:** Which months are peak vs trough? By how much?
4. **Operational inefficiencies:** Are days_to_ship unusually high for certain categories? Is there a processing bottleneck?
5. **Customer behavior patterns:** Do high-value customers prefer certain payment methods? Do they return products less often?

### Presentation Best Practices

- **Lead with the most important insight**, not the most technically interesting one
- **Quantify everything** — "significantly higher" is weaker than "23% higher"
- **Show uncertainty** — if a difference could be explained by sampling variation, say so
- **Use consistent terminology** — if you call it "revenue" in one chart, do not call it "sales" in another
- **Provide context** — a 5% return rate is good or bad only relative to the industry benchmark


In [ ]:
# Insight 1: Revenue & Profit by product_categorycat_summary = df.groupby('product_category').agg(total_revenue=('sales_amount','sum'),total_profit=('profit','sum'),order_count=('order_id','count')).sort_values('total_revenue', ascending=False)fig, axes = plt.subplots(1, 2, figsize=(14, 5))cat_summary['total_revenue'].div(1e6).sort_values().plot(kind='barh', ax=axes[0], color='steelblue')axes[0].set_title('Total Revenue by Category (₹M)')axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:.0f}M'))cat_summary['total_profit'].div(1e6).sort_values().plot(kind='barh', ax=axes[1], color='#2ecc71')axes[1].set_title('Total Profit by Category (₹M)')axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:.0f}M'))plt.tight_layout()plt.show()print(f"Highest Revenue: {cat_summary['total_revenue'].idxmax()}")print(f"Highest Profit: {cat_summary['total_profit'].idxmax()}")

In [ ]:
# Insight 2: Return rate by regionreturn_by_region = df.groupby('region').agg(total_orders=('order_id','count'),returned=('return_flag','sum'))return_by_region['return_rate_pct'] = (return_by_region['returned']/ return_by_region['total_orders'] * 100).round(2)return_by_region = return_by_region.sort_values('return_rate_pct', ascending=False)fig, ax = plt.subplots(figsize=(9, 4))bars = ax.bar(return_by_region.index, return_by_region['return_rate_pct'],color=['#e74c3c' if v == return_by_region['return_rate_pct'].max()else 'steelblue' for v in return_by_region['return_rate_pct']])ax.axhline(return_by_region['return_rate_pct'].mean(), color='orange',linestyle='--', label=f"Avg {return_by_region['return_rate_pct'].mean():.1f}%")for bar in bars:ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,f'{bar.get_height():.1f}%', ha='center', fontsize=10)ax.set_title('Return Rate % by Region'); ax.set_ylabel('Return Rate %')ax.legend()plt.tight_layout()plt.show()print(return_by_region[['total_orders','returned','return_rate_pct']].to_string())

In [ ]:
# Insight 3: Top 10 cities by revenuecity_revenue = df.groupby('city')['sales_amount'].sum().nlargest(10).sort_values()fig, ax = plt.subplots(figsize=(10, 6))city_revenue.plot(kind='barh', color='darkcyan', ax=ax)for i, (city, val) in enumerate(city_revenue.items()):ax.text(val * 1.01, i, f'₹{val/1e6:.2f}M', va='center', fontsize=9)ax.set_title('Top 10 Cities by Total Sales Revenue')ax.set_xlabel('Total Sales (INR)')ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.0f}M'))plt.tight_layout()plt.show()print(f"Top city: {city_revenue.idxmax()} with ₹{city_revenue.max()/1e6:.2f}M in sales")

In [ ]:
# Insight 4: Payment method vs avg order valuepay_stats = df.groupby('payment_method')['sales_amount'].agg(['mean','sem']).round(2)pay_stats.columns = ['mean','sem']pay_stats = pay_stats.sort_values('mean', ascending=False)ci95 = pay_stats['sem'] * 1.96 # 95% CI half-widthfig, ax = plt.subplots(figsize=(10, 5))ax.bar(pay_stats.index, pay_stats['mean'], yerr=ci95, capsize=5,color='darkorchid', alpha=0.8, ecolor='black')ax.set_title('Average Order Value by Payment Method\n(error bars = 95% CI)')ax.set_ylabel('Avg Sales Amount (INR)')ax.set_xticklabels(pay_stats.index, rotation=20, ha='right')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))plt.tight_layout()plt.show()print(f"Highest avg order value: {pay_stats['mean'].idxmax()} (₹{pay_stats['mean'].max():,.0f})")

---
## Assignment 2 — Exploratory Data Analysis — Complete! Well done! Proceed to the next assignment notebook. | Assignment | Notebook File |
|---|---|
| 1. Data Cleaning | `A1_Data_Cleaning.ipynb` |
| 2. EDA | `A2_EDA.ipynb` |
| 3. Statistical Analysis | `A3_Statistical_Analysis.ipynb` |
| 4. Segmentation & Clustering | `A4_Segmentation_Clustering.ipynb` |
| 5. Time Series Forecasting | `A5_Time_Series_Forecasting.ipynb` |